In [3]:
import pandas as pd
import ast

In [4]:
df = pd.read_csv("../../data/raw/movies_metadata.csv", low_memory=False)
df.shape

(45466, 24)

In [5]:
df[[
    "id",
    "original_title",
    "release_date",
    "genres",
    "production_countries",
    "spoken_languages"
]].head()

,id,original_title,release_date,genres,production_countries,spoken_languages
0,862,Toy Story,1995-10-30,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}]"
1,8844,Jumanji,1995-12-15,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}, {'iso..."
2,15602,Grumpier Old Men,1995-12-22,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}]"
3,31357,Waiting to Exhale,1995-12-22,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}]"
4,11862,Father of the Bride Part II,1995-02-10,"[{'id': 35, 'name': 'Comedy'}]","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}]"


In [6]:
df[[
    "id",
    "original_title",
    "genres",
    "production_countries",
    "spoken_languages"
    ]].isna().sum()

id                      0
original_title          0
genres                  0
production_countries    3
spoken_languages        6
dtype: int64

In [7]:
nested_columns = [
    "genres",
    "production_countries",
    "spoken_languages"
]

for column in nested_columns:
    print(column, (df[column] == "[]").sum())

genres 2442
production_countries 6282
spoken_languages 3829


In [8]:
def check_nested_value(value):
    if pd.isna(value):
        return "missing"
    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return "malformed"
    
    if parsed == []:
        return "empty_list"
    
    if not isinstance(parsed, list):
        return "not_a_list"
    
    return "valid"

In [9]:
for column in nested_columns:
    print(column)
    print(df[column].apply(check_nested_value).value_counts())
    print()

genres
genres
valid         43024
empty_list     2442
Name: count, dtype: int64

production_countries
production_countries
valid         39178
empty_list     6282
missing           3
not_a_list        3
Name: count, dtype: int64

spoken_languages
spoken_languages
valid         41631
empty_list     3829
missing           6
Name: count, dtype: int64



In [10]:
country_status = df["production_countries"].apply(check_nested_value)

df.loc[
    country_status == "not_a_list",
    ["id", "original_title", "production_countries"]
]

,id,original_title,production_countries
19730,1997-08-20,"[{'iso_639_1': 'en', 'name': 'English'}]",6.0
29503,2012-09-29,"[{'iso_639_1': 'ja', 'name': '日本語'}]",7.0
35587,2014-01-01,"[{'iso_639_1': 'en', 'name': 'English'}]",4.3


In [11]:
pd.set_option("display.max_columns", None)

df.loc[[19730, 29503, 35587]]

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
19730,- Written by Ørnås,0.065736,/ff9qCepilowshEtG2GYWwzt2bs4.jpg,"[{'name': 'Carousel Productions', 'id': 11176}...","[{'iso_3166_1': 'CA', 'name': 'Canada'}, {'iso...",1997-08-20,0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Midnight Man,False,6.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29503,Rune Balot goes to a casino connected to the ...,1.931659,/zV8bHuSL6WXoD6FWogP9j4x80bL.jpg,"[{'name': 'Aniplex', 'id': 2883}, {'name': 'Go...","[{'iso_3166_1': 'US', 'name': 'United States o...",2012-09-29,0,68.0,"[{'iso_639_1': 'ja', 'name': '日本語'}]",Released,NaN,Mardock Scramble: The Third Exhaust,False,7.0,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35587,Avalanche Sharks tells the story of a bikini ...,2.185485,/zaSf5OG7V8X8gqFvly88zDdRm46.jpg,"[{'name': 'Odyssey Media', 'id': 17161}, {'nam...","[{'iso_3166_1': 'CA', 'name': 'Canada'}]",2014-01-01,0,82.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Beware Of Frost Bites,Avalanche Sharks,False,4.3,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
def check_inner_structure(value):
    if pd.isna(value):
        return "missing"

    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return "malformed"

    if parsed == []:
        return "empty_list"

    if not isinstance(parsed, list):
        return "not_a_list"

    for item in parsed:
        if not isinstance(item, dict):
            return "item_not_dictionary"

        if "name" not in item:
            return "missing_name"

        if not isinstance(item["name"], str) or item["name"].strip() == "":
            return "empty_name"

    return "valid"

In [13]:
for column in nested_columns:
    print(column)
    print(df[column].apply(check_inner_structure).value_counts())
    print()

genres
genres
valid         43024
empty_list     2442
Name: count, dtype: int64

production_countries
production_countries
valid         39178
empty_list     6282
missing           3
not_a_list        3
Name: count, dtype: int64

spoken_languages
spoken_languages
valid         41147
empty_list     3829
empty_name      484
missing           6
Name: count, dtype: int64



In [14]:
language_status = df["spoken_languages"].apply(check_inner_structure)

df.loc[
    language_status == "empty_name",
    ["id", "original_title", "spoken_languages"]
].head(10)

,id,original_title,spoken_languages
108,197,Braveheart,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso..."
151,649,Belle de jour,"[{'iso_639_1': 'fr', 'name': 'Français'}, {'is..."
211,19155,Before the Rain,"[{'iso_639_1': 'sq', 'name': 'shqip'}, {'iso_6..."
262,4476,Legends of the Fall,"[{'iso_639_1': 'kw', 'name': ''}, {'iso_639_1'..."
284,241,Natural Born Killers,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso..."
286,527,Once Were Warriors,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso..."
297,30304,Picture Bride,"[{'iso_639_1': 'tl', 'name': ''}, {'iso_639_1'..."
413,46924,Being Human,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso..."
440,26141,Fatal Instinct,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso..."
504,713,The Piano,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso..."


In [15]:
def extract_names(value):
    if pd.isna(value):
        return pd.NA

    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return pd.NA

    if not isinstance(parsed, list) or not parsed:
        return pd.NA

    names = []

    for item in parsed:
        if isinstance(item, dict):
            name = item.get("name")

            if isinstance(name, str):
                name = name.strip()

                if name and name not in names:
                    names.append(name)

    if not names:
        return pd.NA

    return "|".join(names)

In [17]:
cleaned_df = df[[
    "id",
    "original_title",
    "genres",
    "production_countries",
    "spoken_languages"
]].copy()

for column in nested_columns:
    cleaned_df[column] = cleaned_df[column].apply(extract_names)

cleaned_df.head()

,id,original_title,genres,production_countries,spoken_languages
0,862,Toy Story,Animation|Comedy|Family,United States of America,English
1,8844,Jumanji,Adventure|Fantasy|Family,United States of America,English|Français
2,15602,Grumpier Old Men,Romance|Comedy,United States of America,English
3,31357,Waiting to Exhale,Comedy|Drama|Romance,United States of America,English
4,11862,Father of the Bride Part II,Comedy,United States of America,English
